# DSAIT4335 Recommender Systems
# Final Project

In this project, you will work to build different recommendation models and evaluate the effectiveness of these models through offline experiments. The dataset used for the experiments is **MovieLens100K**, a movie recommendation dataset collected by GroupLens: https://grouplens.org/datasets/movielens/100k/. For more details, check the project description on Brightspace.

# Instruction

The MovieLens100K is already splitted into 80% training and 20% test sets. Along with training and test sets, movies metadata as content information is also provided.

**Expected file structure** for this assignment:   
   
   ```
   RecSysProject/
   ├── training.txt
   ├── test.txt
   ├── movies.txt
   └── codes.ipynb
   ```

**Note:** Be sure to run all cells in each section sequentially, so that intermediate variables and packages are properly carried over to subsequent cells.

**Note** Be sure to run all cells such that the submitted file contains the output of each cell.

**Note** Feel free to add cells if you need more for answering a question.

**Submission:** Answer all the questions in this jupyter-notebook file. Submit this jupyter-notebook file (your answers included) to Brightspace. Change the name of this jupyter-notebook file to your group number: example, group10 -> 10.ipynb.

# Setup

In [2]:
!pip install transformers torch

# you can refer https://huggingface.co/docs/transformers/en/model_doc/bert for various versions of the pre-trained model BERT

In [3]:
# For BERT embeddings (install: pip install transformers torch)
print("Check the status of BERT installation:")

try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    BERT_AVAILABLE = True
    print("BERT libraries loaded successfully!")
    device = torch.device('cpu')
    print(f"Using device: {device}")
except ImportError:
    BERT_AVAILABLE = False
    print("BERT libraries not available. Install with: pip install transformers torch")

Check the status of BERT installation:
BERT libraries loaded successfully!
Using device: cpu


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cosine, correlation
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
import re
import time, math
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


# Load dataset

In [5]:
# loading the training set and test set
columns_name=['user_id','item_id','rating','timestamp']
train_data = pd.read_csv('training.txt', sep='\t', names=columns_name)
test_data = pd.read_csv('test.txt', sep='\t', names=columns_name)

print(f'The training data:')
display(train_data[['user_id','item_id','rating']].head())
print(f'The shape of the training data: {train_data.shape}')
print('--------------------------------')
print(f'The test data:')
display(test_data[['user_id','item_id','rating']].head())
print(f'The shape of the test data: {test_data.shape}')

The training data:


,user_id,item_id,rating
0,1,1,5
1,1,2,3
2,1,3,4
3,1,4,3
4,1,5,3


The shape of the training data: (80000, 4)
--------------------------------
The test data:


,user_id,item_id,rating
0,1,6,5
1,1,10,3
2,1,12,5
3,1,14,5
4,1,17,3


The shape of the test data: (20000, 4)


In [6]:
movies = pd.read_csv('movies.txt',names=['item_id','title','genres','description'],sep='\t')
movies.head()

,item_id,title,genres,description
0,1,Toy Story (1995),"Animation, Children's, Comedy","A group of sentient toys, who pretend to be li..."
1,2,GoldenEye (1995),"Action, Adventure, Thriller","In 1986, MI6 agents James Bond and Alec Trevel..."
2,3,Four Rooms (1995),Thriller,"On New Year's Eve, bellhop Sam (Marc Lawrence)..."
3,4,Get Shorty (1995),"Action, Comedy, Drama",Chili Palmer is a Miami-based loan shark and m...
4,5,Copycat (1995),"Crime, Drama, Thriller",After giving a guest lecture on criminal psych...


# Task 1) Implementation of different recommendation models as well as a hybrid model combining those recommendation models

In [7]:
import os
import torch

def create_bert_embeddings(content, label):
    """
    Generate BERT embeddings for movie content.

    Args:
        content: Content of items

    Returns:
        numpy.ndarray: BERT embeddings matrix
    """
    if os.path.exists(label + ".pt"):
        return torch.load(label + '.pt').detach().cpu().numpy()
    
    if not BERT_AVAILABLE:
        print("BERT libraries not available. Install with: pip install transformers torch")
        return None

    if content is None:
        return None

    if isinstance(content, pd.Series):
        content = content.fillna("").astype(str).tolist()
    elif isinstance(content, np.ndarray):
        content = content.astype(str).tolist()

    model_name = 'distilbert-base-uncased'

    print(f"Loading BERT model: {model_name}")

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Set device (GPU if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using cuda or cpu: {device}")
    model.to(device)
    model.eval()

    print(f"Using device: {device}")

    # Generate embeddings in batches
    batch_size = 32  # Adjust based on available memory
    emb = []

    for i in range(0, len(content), batch_size):
        if i % (batch_size * 10) == 0:
            print(f"Processing batch {i//batch_size + 1}/{len(content)//batch_size + 1}")

        batch_texts = content[i:i + batch_size]

        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )

        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate embeddings
        with torch.no_grad():
            outputs = model(**inputs)

            # Use [CLS] token embedding (first token)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            emb.extend(batch_embeddings)

    emb = np.array(emb)
    torch_emb = torch.from_numpy(emb)

    torch.save(torch_emb, label + '.pt')

    print(f"BERT embeddings generated: {emb.shape}")
    print(f"Embedding dimension: {emb.shape[1]}")

    return emb

In [55]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

class ContentRecommender():
    def __init__(self, movies_df, train_data, tfidf_max_features=20000, svd_dim=256, normalize_emb=True):
        self.train_data = train_data
        
        movies_df = movies_df.copy()
        movies_df['titlegenres'] = movies_df['title'] + ' ' + movies_df['genres']
        movies_df['full'] = movies_df['titlegenres'] + ' ' + movies_df['description'].fillna('')
    
        def compute_tfidf(corpus):
            tfidf_vectorizer = TfidfVectorizer(
                tokenizer=lambda s: re.findall(r'\w+|\S', s.lower()),
                lowercase=False,
                max_features=tfidf_max_features,
                ngram_range=(1,2),
                stop_words='english'
            )
            X = tfidf_vectorizer.fit_transform(corpus)
            svd = TruncatedSVD(n_components=svd_dim, random_state=42)
            X_reduced = svd.fit_transform(X)
            if normalize_emb:
                X_reduced = normalize(X_reduced)
            return X_reduced
    
        tfidf_full = compute_tfidf(movies_df['full'].tolist())
    
        self.item_emb_full = create_bert_embeddings(movies_df['full'].tolist(), 'full')
    
        self.emb_full = np.hstack([self.item_emb_full, tfidf_full])
        if normalize_emb:
            self.emb_full = normalize(self.emb_full)
    
        self.item_to_idx = {iid: idx for idx, iid in enumerate(movies_df['item_id'])}

    def score(self, user_id, movie_id):
        if movie_id not in self.item_to_idx:
            return 0.0
    
        user_ratings = self.train_data[self.train_data["user_id"] == user_id]
        if len(user_ratings) == 0:
            return 0.0
    
        item_indices = [
            self.item_to_idx[iid] for iid in user_ratings["item_id"]
            if iid in self.item_to_idx
        ]
        if len(item_indices) == 0:
            return 0.0
    
        weights = user_ratings[user_ratings["item_id"].isin(self.item_to_idx.keys())]["rating"].values
        item_embs = self.emb_full[item_indices]
    
        user_profile = np.average(item_embs, axis=0, weights=weights)
    
        user_profile_norm = user_profile / (np.linalg.norm(user_profile) + 1e-9)
        target_emb = self.emb_full[self.item_to_idx[movie_id]]
        target_emb_norm = target_emb / (np.linalg.norm(target_emb) + 1e-9)
    
        score = np.dot(user_profile_norm, target_emb_norm)
    
        return float(score)


content_recommender = ContentRecommender(movies, train_data)

# Print example score
example_movie_id = movies['item_id'].iloc[0]
example_user_id = train_data['user_id'].iloc[0]

print(content_recommender.score(example_user_id, example_movie_id))

0.9580696661692306


# Task 2) Experiments for both rating prediction and ranking tasks, and conducting offline evaluation

In [58]:
import math
from tqdm import tqdm

def create_rankings(score_method, movies, training_data, test_data, top_k=10):
    user_ids = test_data["user_id"].unique()

    user_rankings = []
    for user_id in tqdm(user_ids):
        target_list = test_data[test_data["user_id"] == user_id]
        watched = set(training_data[training_data["user_id"] == user_id]["item_id"])

        unsorted_ranking = []
        for movie_id in movies:
            if movie_id in watched:
                continue

            score = score_method(user_id, movie_id)

            unsorted_ranking.append((movie_id, score))
        sorted_ranking = sorted(unsorted_ranking, key=lambda t: t[1], reverse=True)[:top_k]
        sorted_target = [(t[0], t[1]) for t in sorted(list(target_list[["item_id", "rating"]].itertuples(index=False)), key=lambda t: t[1], reverse=True)[:top_k]]

        user_rankings.append((user_id, sorted_ranking, sorted_target))
        
    return user_rankings

def Precision(ground_truth, rec_list):    
    result = 0.0

    count = 0
    prec_sum = 0
    for user_id in range(0, len(ground_truth)):
        n = len(rec_list[user_id])
        count = count + 1

        inter = len(set(rec_list[user_id]).intersection(set(ground_truth[user_id])))

        prec_sum = prec_sum + (100 * (inter / n))

    result = prec_sum / count

    return result

def Recall(ground_truth, rec_list):    
    result = 0.0
    
    count = 0
    prec_sum = 0
    for user_id in range(0, len(ground_truth)):
        count = count + 1

        inter = len(set(rec_list[user_id]).intersection(set(ground_truth[user_id])))

        prec_sum = prec_sum + (100 * (inter / len(ground_truth[user_id])))

    result = prec_sum / count

    return result

def NDCG(ground_truth, rec_list):    
    result = 0.0
    
    count = 0
    prec_sum = 0
    for user_id in range(0, len(ground_truth)):
        count = count + 1

        inter = set(rec_list[user_id]).intersection(set(ground_truth[user_id]))
        n = len(rec_list[user_id])

        dcg = 0
        for j in range(0, len(rec_list[user_id])):
            if rec_list[user_id][j] in inter:
                dcg = dcg + (1 / math.log(j + 2, 2))

        idcg = 0
        for j in range(0, min(len(ground_truth[user_id]), n)):
            idcg = idcg + (1 / math.log(j + 2, 2))
        
        prec_sum = prec_sum + (dcg / idcg)

    result = prec_sum / count

    return result
        
rankings = create_rankings(content_recommender.score, movies["item_id"], train_data, test_data)

ground_truth = [[tup[0] for tup in ranking[2]] for ranking in rankings]
rec_list = [[tup[0] for tup in ranking[1]] for ranking in rankings]

print(Precision(ground_truth, rec_list))
print(Recall(ground_truth, rec_list))
print(NDCG(ground_truth, rec_list))

AttributeError: Can't pickle local object 'create_rankings.<locals>.create_ranking_tuple'

# Task 3) Implement baselines for both rating prediction and ranking tasks, and perform experiments with those baselines

# Task 4) Analysis of recommendation models. Analyzing the coefficients of hybrid model and the success of recommendation models for different users' groups. 

# Task 5) Evaluation of beyond accuracy

In [59]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy

def Diversity(rec_list, item_embeddings):
    diversity_scores = []
    for rec_items in rec_list:
        if len(rec_items) < 2:
            continue
        embs = [item_embeddings[i] for i in rec_items if i in item_embeddings]
        if len(embs) < 2:
            continue
        embs = np.vstack(embs)
        sim_matrix = cosine_similarity(embs)
        np.fill_diagonal(sim_matrix, 0)
        mean_sim = sim_matrix.mean()
        diversity_scores.append(1 - mean_sim)
    return np.mean(diversity_scores) * 100 if diversity_scores else 0.0


def Novelty(rec_list, training_data):
    item_counts = training_data["item_id"].value_counts()
    total = len(training_data)
    prob = item_counts / total

    novelty_scores = []
    for rec_items in rec_list:
        if not rec_items:
            continue
        score = 0
        for i in rec_items:
            if i in prob:
                score += -np.log2(prob[i])
        novelty_scores.append(score / len(rec_items))
    return np.mean(novelty_scores) if novelty_scores else 0.0


def Calibration(rec_list, user_ids, training_data, movies_df):
    user_groups = training_data.groupby("user_id")["item_id"].apply(list).to_dict()
    movie_genres = movies_df.set_index("item_id")["genres"].to_dict()

    def get_genre_dist(items):
        genres = []
        for i in items:
            if i in movie_genres:
                genres.extend(str(movie_genres[i]).split('|'))
        if not genres:
            return {}
        unique, counts = np.unique(genres, return_counts=True)
        dist = dict(zip(unique, counts / sum(counts)))
        return dist

    cal_scores = []
    for user_id, rec_items in zip(user_ids, rec_list):
        user_items = user_groups.get(user_id, [])
        p_user = get_genre_dist(user_items)
        p_rec = get_genre_dist(rec_items)
        if not p_user or not p_rec:
            continue

        all_genres = set(p_user.keys()).union(p_rec.keys())
        p_user_vec = np.array([p_user.get(g, 1e-9) for g in all_genres])
        p_rec_vec = np.array([p_rec.get(g, 1e-9) for g in all_genres])

        p_user_vec /= p_user_vec.sum()
        p_rec_vec /= p_rec_vec.sum()

        kl = entropy(p_user_vec, p_rec_vec)
        cal_score = 1 / (1 + kl)  # bounded [0,1]
        cal_scores.append(cal_score)

    return np.mean(cal_scores) * 100 if cal_scores else 0.0


def Popularity_Fairness(rec_list, training_data):
    rec_counts = {}
    for rec_items in rec_list:
        for i in rec_items:
            rec_counts[i] = rec_counts.get(i, 0) + 1

    rec_freq = np.array(list(rec_counts.values()))
    if len(rec_freq) == 0:
        return 0.0

    gini = np.sum(np.abs(np.subtract.outer(rec_freq, rec_freq))) / (2 * len(rec_freq)**2 * rec_freq.mean())
    fairness = 1 - gini  # higher = fairer
    return fairness * 100

item_id_to_index = {id_: i for i, id_ in enumerate(movies["item_id"])}
item_embeddings_dict = {id_: content_recommender.item_emb_full[item_id_to_index[id_]] for id_ in movies["item_id"]}

rec_list = [[m for (m, s) in r[1]] for r in rankings]
user_ids = [r[0] for r in rankings]

print("Diversity:", Diversity(rec_list, item_embeddings_dict))
print("Novelty:", Novelty(rec_list, train_data))
print("Calibration:", Calibration(rec_list, user_ids, train_data, movies))
print("Popularity Fairness:", Popularity_Fairness(rec_list, train_data))

Diversity: 12.623361954242315
Novelty: 11.105147723368344
Calibration: 7.441438854034844
Popularity Fairness: 21.05863611872467
